In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyOD Detektoren (die von MetaOD empfohlenen instanziieren wir später)
from pyod.models.iforest import IForest
from pyod.models.ocsvm import OCSVM
from pyod.models.lof import LOF
from pyod.models.knn import KNN
from pyod.models.ecod import ECOD
from pyod.models.abod import ABOD
from pyod.models.auto_encoder import AutoEncoder

# Eigene Pipeline
from automl.data.tep import load_tep_splits, TEPSplits
from automl.evaluation.metrics import pr_auc, roc_auc

# MetaOD selbst läuft in einem isolierten Legacy-Stack (siehe Abschnitt 2),
# nicht in diesem Kernel — daher hier kein metaod-Import.
print("Imports OK")

## 1. Daten laden (nur Fault-Free Training)

Für das unsupervised Setting trainieren wir ausschließlich auf Normalbetrieb.
MetaOD bekommt einen Subsample der Trainingsdaten zur Meta-Feature-Extraktion.

In [ ]:
from pathlib import Path
from sklearn.preprocessing import StandardScaler

splits = load_tep_splits("data/")

# Nur echte Sensor-Features (52): faultNumber/simulationRun/sample sind
# Metadaten bzw. Label und dürfen NICHT als Feature in die Detektoren.
df_train = splits.train_fault_free.features
FEATURE_COLS = [c for c in df_train.columns if c.startswith(("xmeas_", "xmv_"))]
assert len(FEATURE_COLS) == 52

# Nur Fault-Free Trainingsdaten — kein Label, kein Faulty
X_train = df_train[FEATURE_COLS].values

# Skalierung nur auf Normaldaten fitten (wird auch für die Detektoren gebraucht)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)

# Subsampling für MetaOD (250k Zeilen -> 5000 reichen für Meta-Features)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train_scaled), size=5000, replace=False)
X_meta = X_train_scaled[idx]

# Für den isolierten MetaOD-Lauf ablegen
Path("artifacts").mkdir(exist_ok=True)
np.save("artifacts/metaod_input.npy", X_meta)

print(f"Features:               {len(FEATURE_COLS)}")
print(f"Trainingsdaten gesamt:  {X_train_scaled.shape}")
print(f"MetaOD-Input (Sample):  {X_meta.shape}  -> artifacts/metaod_input.npy")

## 2. MetaOD — Detektor-Ranking (labellos)

MetaOD extrahiert Meta-Features aus den Daten und empfiehlt per Meta-Learning
ein Ranking von PyOD-Detektoren **inklusive Hyperparametern** (z.B.
`kNN (5, 'median')`) — ohne einen einzigen Label zu benötigen.

**Warum ein separater Prozess?** MetaODs vortrainierte Modelle wurden mit
scikit-learn 0.22.1 gepickelt. Unser moderner Stack (sklearn 1.9, numpy 2.x)
kann die gepickelten RandomForest-Trees nicht laden (geändertes C-Struct-Layout).
Deshalb läuft die MetaOD-Selektion isoliert in einem Legacy-Stack, den `uv` über
die PEP-723-Metadaten in [`scripts/run_metaod_selection.py`](scripts/run_metaod_selection.py)
automatisch bereitstellt. Zurück kommt nur eine Liste von Empfehlungsstrings.

In [ ]:
import json
import subprocess

# MetaOD im isolierten Legacy-Stack laufen lassen (uv liest die Deps aus dem
# PEP-723-Header des Skripts). Dauert beim ersten Mal etwas länger (Env-Setup).
ranking_path = "artifacts/metaod_ranking.json"
result = subprocess.run(
    ["uv", "run", "scripts/run_metaod_selection.py",
     "artifacts/metaod_input.npy", ranking_path, "20"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stdout)
    raise RuntimeError(result.stderr[-2000:])

with open(ranking_path, encoding="utf-8") as fh:
    metaod_ranking = json.load(fh)

print("MetaOD Detektor-Ranking (Top 20):")
for rank, name in enumerate(metaod_ranking, start=1):
    print(f"  {rank:2}. {name}")

## 3. Testdaten laden (mit korrekten Per-Sample-Labels)

**Achtung Labeling:** In den Faulty-Testläufen startet der Fehler erst ab
`sample >= 160`. Die ersten 159 Samples jedes Laufs sind **Normalbetrieb**.
Ein Sample ist also nur dann eine Anomalie, wenn `faultNumber > 0` **und**
`sample >= 160`. (Die fertige `evaluation_dataset()` labelt alle Faulty-Zeilen
als 1 und ist daher hier nicht verwendbar.)

Wegen der Datengröße (~10 Mio. Testpunkte) subsamplen wir **Run-erhaltend**
(ganze Läufe je Fehlertyp) — so bleiben Detection Delay und Per-Fault-Analyse
später korrekt berechenbar. Skaliert wird mit dem **auf Trainingsdaten**
gefitteten Scaler (kein Leak).

In [12]:
FAULT_START_TEST = 160  # Fehler wird im Test-Lauf ab diesem Sample aktiv


def subsample_runs(df, n_runs, seed):
    """Ganze Läufe je Fehlertyp ziehen (Run-Struktur bleibt erhalten)."""
    rng = np.random.default_rng(seed)
    keep = []
    for _, g in df.groupby("faultnumber"):
        runs = g["simulationrun"].unique()
        sel = rng.choice(runs, size=min(n_runs, len(runs)), replace=False)
        keep.append(g[g["simulationrun"].isin(sel)])
    return pd.concat(keep, ignore_index=True)


N_RUNS_TEST = 10  # je Fehlertyp; hochskalieren für finale Zahlen
faulty_test = subsample_runs(splits.test_faulty.features, N_RUNS_TEST, seed=0)
ff_test = subsample_runs(splits.test_fault_free.features, N_RUNS_TEST, seed=1)
test_df = pd.concat([faulty_test, ff_test], ignore_index=True)

# Korrektes Per-Sample-Label
y_test = (
    (test_df["faultnumber"] > 0) & (test_df["sample"] >= FAULT_START_TEST)
).astype(int).values

# Features (52) + Skalierung mit Trainings-Scaler
X_test = test_df[FEATURE_COLS].values
X_test_scaled = scaler.transform(X_test)

# Metadaten für spätere Per-Fault-/Delay-Analyse behalten
meta_test = test_df[["faultnumber", "simulationrun", "sample"]].copy()

print(f"Test-Subsample:   {X_test_scaled.shape}")
print(f"Anomalie-Anteil:  {y_test.mean():.1%}")
print(f"  normal (y=0):   {(y_test == 0).sum():>7}")
print(f"  anomal (y=1):   {(y_test == 1).sum():>7}")

Test-Subsample:   (201600, 52)
Anomalie-Anteil:  79.5%
  normal (y=0):     41400
  anomal (y=1):    160200


## 4. MetaODs Empfehlung: Parser + Training

MetaOD liefert Strings wie `kNN (5, 'median')`. Wir übersetzen sie in echte
PyOD-Objekte. Distanz-/dichtebasierte Detektoren (ABOD/kNN/LOF/OCSVM) skalieren
quadratisch — deshalb fitten wir sie auf einem **Subsample** der Normaldaten
(nicht allen 250k Punkten).

Die Schwelle für FAR/Detection Rate kommt aus den **Trainings-Scores** (Ziel-FAR
1 %) — ohne Blick auf Testlabels.

In [ ]:
import ast
from pyod.models.cof import COF
from pyod.models.hbos import HBOS
from pyod.models.loda import LODA


def build_detector(spec: str):
    """MetaOD-Empfehlungsstring -> PyOD-Detektor (Parameter nach PyOD-Konvention)."""
    head = spec.split("(")[0].split()[0].lower()
    if "(" in spec:
        args = ast.literal_eval("(" + spec.split("(", 1)[1])
        args = args if isinstance(args, tuple) else (args,)
    else:
        args = (int(spec.split()[1]),)  # z.B. "ABOD 5"

    if head == "abod":
        return ABOD(n_neighbors=int(args[0]))
    if head == "knn":
        return KNN(n_neighbors=int(args[0]), method=args[1])
    if head == "lof":
        return LOF(n_neighbors=int(args[0]), metric=args[1])
    if head == "cof":
        return COF(n_neighbors=int(args[0]))
    if head == "iforest":
        return IForest(n_estimators=int(args[0]), max_features=float(args[1]), random_state=0)
    if head == "hbos":
        return HBOS(n_bins=int(args[0]), alpha=float(args[1]))
    if head == "loda":
        return LODA(n_bins=int(args[0]), n_random_cuts=int(args[1]))
    if head == "ocsvm":
        return OCSVM(nu=float(args[0]), kernel=args[1])
    raise ValueError(f"Unbekannte Detektor-Spezifikation: {spec!r}")


# Subsample der Normaldaten zum Fitten der (quadratischen) Detektoren
FIT_SAMPLE_SIZE = 10_000
_fit_idx = np.random.default_rng(7).choice(
    len(X_train_scaled), size=FIT_SAMPLE_SIZE, replace=False
)
X_fit = X_train_scaled[_fit_idx]
print(f"Fit-Subsample: {X_fit.shape}")

# Smoke-Test: Top-1 parsen
_top1 = build_detector(metaod_ranking[0])
print(f"MetaOD Top-1: {metaod_ranking[0]!r} -> {_top1.__class__.__name__}")

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score


def evaluate_detector(spec: str, far_target: float = 0.01) -> dict:
    """Detektor fitten, Test scoren, schwellenbasierte + -unabhängige Metriken."""
    clf = build_detector(spec)
    clf.fit(X_fit)
    scores = clf.decision_function(X_test_scaled)  # höher = anomaler

    # Schwelle aus Trainings-Scores (kein Testlabel-Leak)
    thr = np.quantile(clf.decision_scores_, 1.0 - far_target)
    y_pred = (scores >= thr).astype(int)
    far = (scores[y_test == 0] >= thr).mean()
    det = (scores[y_test == 1] >= thr).mean()

    return {
        "model": spec,
        "pr_auc": average_precision_score(y_test, scores),
        "roc_auc": roc_auc_score(y_test, scores),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "far": far,
        "detection_rate": det,
        "threshold": thr,
        "_scores": scores,
    }


# MetaODs Top-1 evaluieren
res_top1 = evaluate_detector(metaod_ranking[0])
print(f"MetaOD Top-1: {res_top1['model']}")
print(f"  PR-AUC:          {res_top1['pr_auc']:.3f}")
print(f"  ROC-AUC:         {res_top1['roc_auc']:.3f}")
print(f"  F1:              {res_top1['f1']:.3f}  (bei Ziel-FAR-Schwelle; Imbalance beachten)")
print(f"  FAR:             {res_top1['far']:.3f}  (Ziel 0.01)")
print(f"  Detection Rate:  {res_top1['detection_rate']:.3f}")

## 5. Per-Fault-Analyse

Die gepoolte Detection Rate mittelt über alle 20 Fehler und versteckt, *welche*
Fehler erkannt werden. Pro Fehlertyp berechnen wir:

- **Detection Rate** — Anteil erkannter Fehlersamples (`sample >= 160`)
- **FAR** — Fehlalarme auf der Vor-Fehler-Phase (`sample < 160`) desselben Laufs
- **Detection Delay** — Samples vom Fehlereintritt bis zur ersten Erkennung
  (je Lauf); groß = spät erkannt, `NaN` = im Lauf nie erkannt

Delay und Detection Rate zusammen sind entscheidend: ein Fehler kann am Ende
gut erkannt sein (hohe Rate), aber viel zu spät (großer Delay).

In [ ]:
def per_fault_table(meta, scores, threshold, fault_start=FAULT_START_TEST):
    """Detection Rate, FAR und mittlerer Detection Delay je Fehlertyp."""
    df = meta.copy()
    df["pred"] = (scores >= threshold).astype(int)

    rows = []
    for fault, g in df.groupby("faultnumber"):
        if fault == 0:  # fault-free Läufe hier überspringen
            continue
        fault_pts = g[g["sample"] >= fault_start]
        pre_pts = g[g["sample"] < fault_start]

        delays = []
        for _, run in fault_pts.groupby("simulationrun"):
            run = run.sort_values("sample")
            hits = run[run["pred"] == 1]
            delays.append(int(hits["sample"].iloc[0]) - fault_start if len(hits) else np.nan)

        rows.append({
            "fault": int(fault),
            "detection_rate": fault_pts["pred"].mean(),
            "far": pre_pts["pred"].mean() if len(pre_pts) else np.nan,
            "mean_delay": np.nanmean(delays) if np.any(~np.isnan(delays)) else np.nan,
            "never_detected": int(np.isnan(delays).sum()),
        })
    return pd.DataFrame(rows).set_index("fault")


pf = per_fault_table(meta_test, res_top1["_scores"], res_top1["threshold"])

# Interessante Fehler hervorheben
print("=== Fehler 1, 4, 13 ===")
display(pf.loc[[1, 4, 13]].round(3))

print("\n=== Alle Fehler, sortiert nach Detection Rate (schwerste zuerst) ===")
display(pf.sort_values("detection_rate").round(3))

## 6. Top-k-Schleife: MetaODs ganze Empfehlungsliste bewerten

Bisher nur Top-1. Jetzt trainieren und bewerten wir MetaODs **Top-k**
Empfehlungen. Das liefert die Tabelle *„MetaOD-Rang vs. tatsächliche
Performance"* — und ist die Grundlage für den späteren Oracle-Vergleich
(kommt MetaODs Wahl an den *besten* Detektor der Liste heran?).

Die Ergebnisse werden als CSV in `artifacts/` gespeichert, damit sie sich mit
den EM/MV- und Oracle-Ergebnissen (separate Arbeit) zusammenführen lassen.

> **Hinweis Laufzeit:** ABOD/LOF skalieren quadratisch — die Schleife über
> mehrere solcher Detektoren dauert ein paar Minuten.

In [ ]:
import time

TOP_K = 10  # wie viele der MetaOD-Empfehlungen durchrechnen

results = []
for rank, spec in enumerate(metaod_ranking[:TOP_K], start=1):
    t0 = time.time()
    res = evaluate_detector(spec)
    res["metaod_rank"] = rank
    results.append(res)
    print(f"{rank:2}. {spec:26} PR-AUC={res['pr_auc']:.3f} "
          f"ROC={res['roc_auc']:.3f} FAR={res['far']:.3f} "
          f"Det={res['detection_rate']:.3f}  ({time.time()-t0:.0f}s)")

# Vergleichstabelle (ohne die großen _scores-Arrays)
metrics_df = pd.DataFrame(
    [{k: v for k, v in r.items() if not k.startswith("_")} for r in results]
)
metrics_df = metrics_df[
    ["metaod_rank", "model", "pr_auc", "roc_auc", "f1", "far", "detection_rate", "threshold"]
]

print("\n=== MetaOD Top-k: Rang vs. Performance ===")
display(metrics_df.round(3))

# Bester Detektor der Liste nach PR-AUC (Vorschau auf den Oracle-Gedanken)
best = metrics_df.loc[metrics_df["pr_auc"].idxmax()]
print(f"\nMetaOD-Rang 1:           {metrics_df.iloc[0]['model']}  "
      f"(PR-AUC {metrics_df.iloc[0]['pr_auc']:.3f})")
print(f"Bester der Liste (PR-AUC): {best['model']}  auf MetaOD-Rang {int(best['metaod_rank'])}  "
      f"(PR-AUC {best['pr_auc']:.3f})")

## 7. Ergebnisse speichern (für den Merge mit EM/MV + Oracle)

Wir legen zwei Artefakte ab:

- `metaod_topk_metrics.csv` — die Vergleichstabelle (ein Detektor je Zeile)
- `metaod_topk_per_fault.csv` — Per-Fault Detection Rate für alle bewerteten
  Detektoren (long format), für die spätere Fehlertyp-Analyse

Die CSVs sind das Übergabeformat an die EM/MV-/Oracle-Arbeit. **Wichtig für den
Merge:** gleiche Testdaten (`N_RUNS_TEST`, Seeds), gleiche FAR-Schwelle,
gleiche Metriknamen — sonst passen die Zahlen nicht zusammen.

In [ ]:
# Konfiguration mitspeichern, damit der Merge reproduzierbar bleibt
run_config = {
    "n_runs_test": N_RUNS_TEST,
    "fit_sample_size": FIT_SAMPLE_SIZE,
    "far_target": 0.01,
    "fault_start_test": FAULT_START_TEST,
    "test_anomaly_ratio": float(y_test.mean()),
    "top_k": TOP_K,
}

# 1) Vergleichstabelle
metrics_df.to_csv("artifacts/metaod_topk_metrics.csv", index=False)

# 2) Per-Fault Detection Rate für alle bewerteten Detektoren (long format)
per_fault_rows = []
for res in results:
    pf_i = per_fault_table(meta_test, res["_scores"], res["threshold"])
    pf_i = pf_i.reset_index()
    pf_i.insert(0, "model", res["model"])
    pf_i.insert(0, "metaod_rank", res["metaod_rank"])
    per_fault_rows.append(pf_i)

per_fault_df = pd.concat(per_fault_rows, ignore_index=True)
per_fault_df.to_csv("artifacts/metaod_topk_per_fault.csv", index=False)

with open("artifacts/metaod_run_config.json", "w", encoding="utf-8") as fh:
    json.dump(run_config, fh, indent=2)

print("Gespeichert in artifacts/:")
print(f"  metaod_topk_metrics.csv     ({len(metrics_df)} Detektoren)")
print(f"  metaod_topk_per_fault.csv   ({len(per_fault_df)} Zeilen)")
print(f"  metaod_run_config.json")
print(f"\nRun-Config: {run_config}")